# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print summary information
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs. Record sets and fields are referenced by their `@id` for precise access.

In [ ]:
# List all available record sets and associated field @ids
record_sets = dataset.record_sets

print("Record Sets available in dataset:")
for rs in record_sets:
    print(f"- Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id})")
    print("\n")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
# Use the @id for record sets

# Gather record set @ids
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

# Show columns for the first populated record set
if dataframes:
    first_rs_id = next(iter(dataframes))
    print(f"Columns for record set {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()
else:
    print("No record sets were populated with data records.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalization, and grouping. Refer to fields by their `@id` for precise handling.

In [ ]:
# Example EDA on the first record set available
if dataframes:
    # Pick the first record set for demonstration
    record_set_id = first_rs_id
    df = dataframes[record_set_id]

    # Find a numeric field. We'll try 'phq9_total_score', which may be present as a column if PHQ-9 scores are recorded.
    numeric_field_id = None
    for col in df.columns:
        if 'phq9_total_score' in col.lower():
            numeric_field_id = col
            break
    # Or fallback to first numeric-looking column
    if not numeric_field_id:
        for col in df.select_dtypes(include='number').columns:
            numeric_field_id = col
            break

    if numeric_field_id:
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Try grouping by a demographic field, e.g. 'gender'
        group_field_id = None
        for col in df.columns:
            if 'gender' in col.lower():
                group_field_id = col
                break

        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No demographic grouping field (e.g. 'gender') found in columns.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualization of numeric field distribution
import matplotlib.pyplot as plt

if dataframes and numeric_field_id:
    df = dataframes[record_set_id]
    plt.figure(figsize=(8,5))
    df[numeric_field_id].hist(bins=20, color='skyblue', edgecolor='black')
    plt.title(f'Histogram of {numeric_field_id} values')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Example scatter plot: numeric field vs group_field if available
    if group_field_id:
        # Plot boxplot by group_field
        plt.figure(figsize=(7,5))
        df.boxplot(column=numeric_field_id, by=group_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.suptitle('')
        plt.ylabel(numeric_field_id)
        plt.xlabel(group_field_id)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Mental Health Survey Dataset from Kilifi offers demographic and psychological survey data ready for AI research and community health analysis.
- Many fields, including PHQ-9 scores and participant demographics, are referenced precisely via their `@id`.
- Common EDA tasks—including filtering, normalization, and grouping—can be conducted directly after loading records with `mlcroissant`.
- Visualizations highlight data distributions and relationships, enabling actionable insights for public health and research.
- For more details, see [mlcroissant documentation](https://mlcommons.github.io/croissant/).
